# 01 · Evaluation Pitfalls & Correct Data Splits

## What this notebook covers
Data leakage is the single most common reason a model looks great in experiments but fails in production. This notebook maps the leakage taxonomy, gives runnable demos of each failure mode, and shows the correct fix side-by-side.

## Why it matters
A 2021 Nature Medicine meta-analysis of 300 clinical ML studies found evidence of leakage in a majority. The pattern repeats across every domain: the model "learns" the test set, reported AUC inflates, and the product misbehaves on live data. Understanding the taxonomy lets you build evaluation pipelines that are honest by construction.

## The leakage taxonomy
| Type | Root cause | Symptom |
|------|-----------|---------|
| Temporal | Future data mixed into training | AUC drops sharply at production cutoff |
| Group | Correlated samples split across train/test | Over-optimistic cross-validation |
| Target / label | Target-derived features used as inputs | Perfect in-sample, near-chance OOD |
| Pipeline | Preprocessing fitted on full dataset then split | Overly narrow confidence intervals |
| Duplicate | Near-duplicate rows in train and test | Memorisation rather than generalisation |

The sections below demonstrate each with synthetic data and show the corrected version.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import (
    train_test_split, cross_val_score,
    GroupKFold, TimeSeriesSplit
)
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
np.random.seed(42)

## 1 · Temporal leakage
The classic mistake: `train_test_split(shuffle=True)` on a time-ordered dataset.
The model can train on Tuesday's data and "predict" Monday — information from the future leaks into the past.

In [ ]:
# Synthetic time-series: 3 years of daily data
n = 1000
dates = pd.date_range("2021-01-01", periods=n, freq="D")
trend = np.linspace(0, 2, n)
noise = np.random.randn(n)
signal = trend + noise
# Label: 1 if next-day signal is higher (only knowable in the future)
labels = (np.roll(signal, -1) > signal).astype(int)
labels[-1] = 0

df = pd.DataFrame({"date": dates, "signal": signal, "label": labels})
df["feat_lag1"] = df["signal"].shift(1).fillna(0)
df["feat_lag7"] = df["signal"].shift(7).fillna(0)

X = df[["signal", "feat_lag1", "feat_lag7"]].values
y = df["label"].values

clf = GradientBoostingClassifier(n_estimators=50, random_state=0)

# ❌ Shuffled split (leaks future into past)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, shuffle=True, random_state=0)
clf.fit(X_tr, y_tr)
auc_leaky = roc_auc_score(y_te, clf.predict_proba(X_te)[:, 1])

# ✅ Temporal split (respects time order)
cutoff = int(n * 0.8)
X_tr_t, X_te_t = X[:cutoff], X[cutoff:]
y_tr_t, y_te_t = y[:cutoff], y[cutoff:]
clf.fit(X_tr_t, y_tr_t)
auc_clean = roc_auc_score(y_te_t, clf.predict_proba(X_te_t)[:, 1])

print(f"Shuffled (leaky) AUC : {auc_leaky:.4f}")
print(f"Temporal (correct) AUC: {auc_clean:.4f}")
print(f"Inflation from leakage: {auc_leaky - auc_clean:.4f}")

## 2 · Group leakage
When samples are correlated by group (patient, store, user) and groups span train and test, the model exploits within-group correlations that won't generalise to unseen groups. Use `GroupKFold`.

In [ ]:
def detect_temporal_leakage(df, date_col, target_col, feature_cols, n_splits=5):
    """Returns (leaky_auc, clean_auc) using shuffle vs TimeSeriesSplit."""
    X = df[feature_cols].values
    y = df[target_col].values
    clf = GradientBoostingClassifier(n_estimators=50, random_state=0)

    leaky = cross_val_score(clf, X, y, cv=5, scoring="roc_auc").mean()

    tscv = TimeSeriesSplit(n_splits=n_splits)
    clean = cross_val_score(clf, X, y, cv=tscv, scoring="roc_auc").mean()
    return leaky, clean

leaky_auc, clean_auc = detect_temporal_leakage(
    df, "date", "label", ["signal", "feat_lag1", "feat_lag7"]
)
print(f"5-fold CV (leaky): {leaky_auc:.4f}")
print(f"TimeSeriesSplit  : {clean_auc:.4f}")

In [ ]:
# Group leakage demo: patient data
n_patients = 50
n_records  = 500
patient_ids = np.random.randint(0, n_patients, n_records)
# Patients have a latent health level that dominates the label
patient_health = {p: np.random.randn() for p in range(n_patients)}
X_g = np.column_stack([
    [patient_health[p] + 0.3 * np.random.randn() for p in patient_ids],
    np.random.randn(n_records)
])
y_g = (np.array([patient_health[p] for p in patient_ids]) > 0).astype(int)

clf2 = GradientBoostingClassifier(n_estimators=50, random_state=0)

# ❌ Random CV ignores patient groups
auc_random = cross_val_score(clf2, X_g, y_g, cv=5, scoring="roc_auc").mean()

# ✅ GroupKFold keeps each patient in one fold
gkf = GroupKFold(n_splits=5)
auc_group = cross_val_score(clf2, X_g, y_g, groups=patient_ids, cv=gkf, scoring="roc_auc").mean()

print(f"Random CV (leaky) AUC : {auc_random:.4f}")
print(f"GroupKFold (correct) AUC: {auc_group:.4f}")

def detect_group_leakage(X, y, groups, n_splits=5):
    """Returns (random_auc, group_auc). Large gap ⟹ group leakage present."""
    clf = GradientBoostingClassifier(n_estimators=50, random_state=0)
    random_auc = cross_val_score(clf, X, y, cv=n_splits, scoring="roc_auc").mean()
    gkf = GroupKFold(n_splits=n_splits)
    group_auc = cross_val_score(clf, X, y, groups=groups, cv=gkf, scoring="roc_auc").mean()
    return random_auc, group_auc

## 3 · Pipeline leakage
Fitting a scaler (or any transformer) on the entire dataset before splitting passes statistics from the test set into training. Always compose transformations inside a `Pipeline` that is fitted only on training folds.

In [ ]:
X_p, y_p = make_classification(n_samples=500, n_features=20, random_state=0)

# ❌ Scaler fitted on full dataset before CV
scaler = StandardScaler()
X_scaled_full = scaler.fit_transform(X_p)
auc_leaky_pipe = cross_val_score(
    GradientBoostingClassifier(n_estimators=50, random_state=0),
    X_scaled_full, y_p, cv=5, scoring="roc_auc"
).mean()

# ✅ Scaler inside Pipeline — fitted only on train fold each iteration
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", GradientBoostingClassifier(n_estimators=50, random_state=0)),
])
auc_clean_pipe = cross_val_score(pipe, X_p, y_p, cv=5, scoring="roc_auc").mean()

print(f"Pre-scaled (leaky) AUC : {auc_leaky_pipe:.4f}")
print(f"Pipeline (correct) AUC : {auc_clean_pipe:.4f}")
print("Pipeline leakage is subtle — the gap is small but real for high-variance transformers.")

## Summary
| Leakage type | Fix |
|---|---|
| Temporal | `TimeSeriesSplit` or hard cutoff |
| Group | `GroupKFold` / `GroupShuffleSplit` |
| Pipeline | Wrap transformers in `sklearn.pipeline.Pipeline` |
| Target | Audit feature provenance; remove anything derived from the label |
| Duplicate | Hash rows; deduplicate before splitting |

The `detect_temporal_leakage` and `detect_group_leakage` utilities above can be dropped into any evaluation harness as a first-pass sanity check.